# Trabajo Práctico: Búsqueda Voraz y A* sobre un Grafo Dirigido
**Curso:** Inteligencia Artificial / Data Science · Semana 3  
**Estudiante:** Alumno  
**Tema:** Estrategias de Búsqueda en Espacios de Estados (UCS, Voraz Best-First, A*)  
**Entregable:** Notebook interactivo y ejecutable con trazas, comparativa y análisis conceptual

---

## 1. Propósito y Fundamentos Arquitectónicos

El objetivo fundamental de este trabajo **no es simplemente codificar tres funciones para que "den algo"**, sino contrastar rigurosamente dos dimensiones críticas en algoritmos de búsqueda:
1. **Esfuerzo computacional (rapidez):** medido en estados generados, estados expandidos y tamaño de la frontera.
2. **Garantía de optimalidad:** la promesa formal matemática de que la solución devuelta es la de costo mínimo global.

### El Grafo del Problema
Formalizamos el problema como una tupla $P = (S, A, T, s_0, G, c)$:
* **Conjunto de estados:** $\{S, A, B, C, D, G\}$
* **Estado inicial:** $s_0 = S$
* **Estado objetivo:** $G = \{G\}$
* **Transiciones dirigidas y costos:**
  * $S \to A$ ($c=2$), $S \to B$ ($c=2$)
  * $A \to C$ ($c=2$), $A \to D$ ($c=5$)
  * $B \to D$ ($c=2$)
  * $C \to G$ ($c=3$)
  * $D \to G$ ($c=6$)
* **Heurística $h(n)$ (estimación del costo restante a $G$):**
  * $h(S)=7, h(A)=5, h(B)=7, h(C)=3, h(D)=6, h(G)=0$

```
          [2]           [2]           [3]
     (S) -----> (A) ----------> (C) -------> ((G))
      |          |                            ^
      | [2]      | [5]                        | [6]
      v          v                            |
     (B) -----> (D) --------------------------+
          [2]
```


In [1]:
# ==============================================================================
# 1. REPRESENTACIÓN FORMAL DEL GRAFO Y HEURÍSTICA
# ==============================================================================

# Grafo dirigido representado como lista de adyacencia con pesos
GRAFO = {
    'S': [('A', 2), ('B', 2)],
    'A': [('C', 2), ('D', 5)],
    'B': [('D', 2)],
    'C': [('G', 3)],
    'D': [('G', 6)],
    'G': []
}

# Diccionario de estimación heurística h(n) hacia el objetivo 'G'
HEURISTICA = {
    'S': 7,
    'A': 5,
    'B': 7,
    'C': 3,
    'D': 6,
    'G': 0
}

ESTADO_INICIAL = 'S'
ESTADO_OBJETIVO = 'G'

print("Representación cargada exitosamente:")
print(f"Estados: {list(GRAFO.keys())}")
print(f"Inicio: {ESTADO_INICIAL} -> Objetivo: {ESTADO_OBJETIVO}")


Representación cargada exitosamente:
Estados: ['S', 'A', 'B', 'C', 'D', 'G']
Inicio: S -> Objetivo: G


## 2. Estructura de Datos y Arquitectura del Esquema Común

Para evitar duplicación de lógica y asegurar una comparación científicamente justa, los tres algoritmos comparten exactamente la misma estructura de datos y el mismo ciclo de control (*Best-First Search* generalizado).

### Componentes Clave:
1. **Representación del Nodo:** Cada nodo almacena `(estado, padre, accion, g, h, f)`:
   * $g$: Costo exacto acumulado desde la raíz $S$ hasta este nodo.
   * $h$: Estimación heurística desde este estado hasta la meta $G$.
   * $f$: Prioridad efectiva según el algoritmo:
     * **UCS:** $f = g$
     * **Voraz (Greedy):** $f = h$
     * **A\*:** $f = g + h$
2. **Cola de Prioridad con Desempate Determinista (FIFO):**
   * Usamos `heapq`. Los elementos insertados son tuplas `(prioridad_f, orden_insercion, nodo)`.
   * Si dos nodos tienen idéntico valor de $f$, Python desempata por el segundo campo `orden_insercion` (un contador entero estrictamente ascendente). Esto garantiza de forma robusta la regla FIFO exigida sin requerir sobrecargas de operadores en el diccionario del nodo.
3. **Diccionario de Control de Costos (`mejor_g`):**
   * Almacena el menor costo $g$ con el que se ha alcanzado cada estado.
   * **Relajación:** Solo se inserta un sucesor si su nuevo costo $g$ es estrictamente menor al registrado previamente (`nuevo_g < mejor_g[sucesor]`). Si ya existía, se contabiliza como **reapertura**.
4. **Descarte de Nodos Obsoletos (*Lazy Deletion*):**
   * Si un nodo fue superado mientras esperaba en la cola de prioridad (`nodo.g > mejor_g[nodo.estado]`), se descarta silenciosamente al ser extraído.
5. **Prueba de Meta al Extraer:**
   * **REGLA DE ORO:** La meta se comprueba cuando el nodo es **extraído** de la frontera con `heappop()`, **nunca al generarlo**. Verificar la meta al generar invalidaría la garantía de optimalidad de UCS y A*.


In [2]:
import heapq
from typing import Dict, List, Tuple, Any, Optional

def crear_nodo(estado: str, padre: Optional[Dict[str, Any]] = None, 
               accion: Optional[str] = None, g: float = 0, h: float = 0, f: float = 0) -> Dict[str, Any]:
    """Crea la estructura de datos que representa un nodo en el árbol de búsqueda."""
    return {
        'estado': estado,
        'padre': padre,
        'accion': accion,
        'g': g,
        'h': h,
        'f': f
    }

def reconstruir_camino(nodo_meta: Dict[str, Any]) -> List[str]:
    """Reconstruye la secuencia de estados siguiendo los punteros al padre hasta la raíz."""
    camino = []
    actual = nodo_meta
    while actual is not None:
        camino.append(actual['estado'])
        actual = actual['padre']
    return camino[::-1]

def resolver_busqueda(grafo: Dict[str, List[Tuple[str, int]]],
                      heuristica: Dict[str, int],
                      inicio: str,
                      objetivo: str,
                      estrategia: str,
                      verbose: bool = True) -> Dict[str, Any]:
    """
    Motor unificado de búsqueda en grafos para UCS, Voraz y A*.
    Estrategias soportadas: 'UCS', 'VORAZ', 'A*'
    """
    estrategia = estrategia.upper()
    if estrategia not in ['UCS', 'VORAZ', 'A*']:
        raise ValueError(f"Estrategia '{estrategia}' no reconocida. Usar: 'UCS', 'VORAZ', 'A*'")

    # Calcular prioridad f inicial según la estrategia
    g_ini = 0
    h_ini = heuristica[inicio]
    if estrategia == 'UCS':
        f_ini = g_ini
    elif estrategia == 'VORAZ':
        f_ini = h_ini
    else:  # A*
        f_ini = g_ini + h_ini

    nodo_raiz = crear_nodo(inicio, padre=None, accion=None, g=g_ini, h=h_ini, f=f_ini)

    # Frontera: heap de tuplas (f, contador_insercion, nodo)
    contador_insercion = 0
    frontera = []
    heapq.heappush(frontera, (nodo_raiz['f'], contador_insercion, nodo_raiz))
    contador_insercion += 1

    # Tabla de mejor g conocido por estado
    mejor_g = {inicio: 0}

    # Métricas del algoritmo
    estados_generados = 1  # La raíz
    estados_expandidos = 0
    max_tam_frontera = 1
    reaperturas = 0
    traza = []

    paso = 0

    if verbose:
        print(f"\n{'='*70}")
        print(f" INICIANDO BÚSQUEDA: {estrategia}")
        print(f"{'='*70}")

    while frontera:
        max_tam_frontera = max(max_tam_frontera, len(frontera))
        prio_f, _, nodo_actual = heapq.heappop(frontera)
        estado_actual = nodo_actual['estado']

        # Descarte de nodos obsoletos (Lazy Deletion)
        if estrategia in ['UCS', 'A*'] and nodo_actual['g'] > mejor_g[estado_actual]:
            if verbose:
                print(f"  [Descarte Obsoleto] Estado '{estado_actual}' con g={nodo_actual['g']} superado por mejor_g={mejor_g[estado_actual]}")
            continue

        paso += 1

        # Captura de estado de frontera para traza
        items_frontera = [(f, n['estado'], n['g'], n['h']) for f, _, n in sorted(frontera)]
        info_paso = {
            'paso': paso,
            'extrae': estado_actual,
            'g': nodo_actual['g'],
            'h': nodo_actual['h'],
            'f': nodo_actual['f'],
            'frontera_restante': items_frontera
        }
        traza.append(info_paso)

        if verbose:
            str_frontera = ", ".join([f"{e}(f={f},g={g},h={h})" for f, e, g, h in items_frontera]) or "vacía"
            print(f"Paso {paso}: Extrae '{estado_actual}' [f={nodo_actual['f']}, g={nodo_actual['g']}, h={nodo_actual['h']}] | Frontera: [{str_frontera}]")

        # Comprobación de objetivo AL EXTRAER
        if estado_actual == objetivo:
            camino = reconstruir_camino(nodo_actual)
            costo_total = nodo_actual['g']
            if verbose:
                print(f"\n -> ¡META ALCANZADA! Camino: {' -> '.join(camino)} | Costo total: {costo_total}")
            return {
                'algoritmo': estrategia,
                'camino': camino,
                'costo': costo_total,
                'expandidos': estados_expandidos,
                'generados': estados_generados,
                'max_frontera': max_tam_frontera,
                'reaperturas': reaperturas,
                'traza': traza,
                'nodo_meta': nodo_actual
            }

        # Expansión del nodo actual
        estados_expandidos += 1

        for sucesor, costo_arista in grafo.get(estado_actual, []):
            nuevo_g = nodo_actual['g'] + costo_arista
            nuevo_h = heuristica[sucesor]
            estados_generados += 1

            # Calcular nueva prioridad según estrategia
            if estrategia == 'UCS':
                nuevo_f = nuevo_g
            elif estrategia == 'VORAZ':
                nuevo_f = nuevo_h
            else:  # A*
                nuevo_f = nuevo_g + nuevo_h

            # Criterio de relajación e inserción
            if sucesor not in mejor_g or nuevo_g < mejor_g[sucesor]:
                if sucesor in mejor_g:
                    reaperturas += 1
                    if verbose:
                        print(f"    * Reapertura de '{sucesor}': g anterior={mejor_g[sucesor]} -> nuevo g={nuevo_g}")
                
                mejor_g[sucesor] = nuevo_g
                nuevo_nodo = crear_nodo(
                    estado=sucesor,
                    padre=nodo_actual,
                    accion=f"{estado_actual} -> {sucesor}",
                    g=nuevo_g,
                    h=nuevo_h,
                    f=nuevo_f
                )
                heapq.heappush(frontera, (nuevo_f, contador_insercion, nuevo_nodo))
                contador_insercion += 1
                if verbose:
                    print(f"    + Inserta '{sucesor}' con f={nuevo_f} (g={nuevo_g}, h={nuevo_h})")
            else:
                if verbose:
                    print(f"    - Poda de '{sucesor}': nuevo g={nuevo_g} >= mejor_g={mejor_g[sucesor]}")

    raise RuntimeError("No se encontró camino a la meta (frontera agotada).")


## 3. Ejecución y Registro de Trazas Detalladas

A continuación ejecutamos cada algoritmo de manera individual, registrando la secuencia completa de extracciones, decisiones de poda/reapertura y evolución de la frontera.


In [3]:
res_ucs = resolver_busqueda(GRAFO, HEURISTICA, ESTADO_INICIAL, ESTADO_OBJETIVO, 'UCS')
res_voraz = resolver_busqueda(GRAFO, HEURISTICA, ESTADO_INICIAL, ESTADO_OBJETIVO, 'VORAZ')
res_astar = resolver_busqueda(GRAFO, HEURISTICA, ESTADO_INICIAL, ESTADO_OBJETIVO, 'A*')



 INICIANDO BÚSQUEDA: UCS
Paso 1: Extrae 'S' [f=0, g=0, h=7] | Frontera: [vacía]
    + Inserta 'A' con f=2 (g=2, h=5)
    + Inserta 'B' con f=2 (g=2, h=7)
Paso 2: Extrae 'A' [f=2, g=2, h=5] | Frontera: [B(f=2,g=2,h=7)]
    + Inserta 'C' con f=4 (g=4, h=3)
    + Inserta 'D' con f=7 (g=7, h=6)
Paso 3: Extrae 'B' [f=2, g=2, h=7] | Frontera: [C(f=4,g=4,h=3), D(f=7,g=7,h=6)]
    * Reapertura de 'D': g anterior=7 -> nuevo g=4
    + Inserta 'D' con f=4 (g=4, h=6)
Paso 4: Extrae 'C' [f=4, g=4, h=3] | Frontera: [D(f=4,g=4,h=6), D(f=7,g=7,h=6)]
    + Inserta 'G' con f=7 (g=7, h=0)
Paso 5: Extrae 'D' [f=4, g=4, h=6] | Frontera: [D(f=7,g=7,h=6), G(f=7,g=7,h=0)]
    - Poda de 'G': nuevo g=10 >= mejor_g=7
  [Descarte Obsoleto] Estado 'D' con g=7 superado por mejor_g=4
Paso 6: Extrae 'G' [f=7, g=7, h=0] | Frontera: [vacía]

 -> ¡META ALCANZADA! Camino: S -> A -> C -> G | Costo total: 7

 INICIANDO BÚSQUEDA: VORAZ
Paso 1: Extrae 'S' [f=7, g=0, h=7] | Frontera: [vacía]
    + Inserta 'A' con f=5 (g=2, h

## 4. Tabla Comparativa de Resultados

Presentamos la tabla formal de resultados requerida en la Sección 5 de la consigna:


In [4]:
import pandas as pd

tabla_comparativa = pd.DataFrame({
    'Métrica / Propiedad': [
        'Camino devuelto',
        'Costo total del camino',
        'Fórmula de prioridad (f)',
        'Estados expandidos antes de extraer G',
        'Estados generados',
        'Tamaño máximo de frontera',
        'Cantidad de reaperturas',
        'Garantía teórica de optimalidad'
    ],
    'UCS (Costo Uniforme)': [
        ' -> '.join(res_ucs['camino']),
        res_ucs['costo'],
        'f = g',
        res_ucs['expandidos'],
        res_ucs['generados'],
        res_ucs['max_frontera'],
        res_ucs['reaperturas'],
        'Sí (siempre óptimo con c >= 0)'
    ],
    'Voraz (Best-First)': [
        ' -> '.join(res_voraz['camino']),
        res_voraz['costo'],
        'f = h',
        res_voraz['expandidos'],
        res_voraz['generados'],
        res_voraz['max_frontera'],
        res_voraz['reaperturas'],
        'No (carece de garantía global)'
    ],
    'A*': [
        ' -> '.join(res_astar['camino']),
        res_astar['costo'],
        'f = g + h',
        res_astar['expandidos'],
        res_astar['generados'],
        res_astar['max_frontera'],
        res_astar['reaperturas'],
        'Sí (óptimo con h admisible)'
    ]
})

# Visualización limpia
display(tabla_comparativa)


,Métrica / Propiedad,UCS (Costo Uniforme),Voraz (Best-First),A*
0,Camino devuelto,S -> A -> C -> G,S -> A -> C -> G,S -> A -> C -> G
1,Costo total del camino,7,7,7
2,Fórmula de prioridad (f),f = g,f = h,f = g + h
3,Estados expandidos antes de extraer G,5,3,3
4,Estados generados,8,6,6
5,Tamaño máximo de frontera,3,3,3
6,Cantidad de reaperturas,1,0,0
7,Garantía teórica de optimalidad,Sí (siempre óptimo con c >= 0),No (carece de garantía global),Sí (óptimo con h admisible)


## 5. Preguntas de Análisis Teórico y Discusión Conceptual

---

### Pregunta 1: ¿Por qué Voraz y A* coinciden en este grafo? ¿Qué condición del grafo y de la heurística lo explica?

**Respuesta Técnica:**
Coinciden porque a lo largo de la trayectoria óptima $S \to A \to C \to G$, la función heurística $h(n)$ es **perfecta (exacta)** respecto al costo real restante $h^*(n)$:
* En $S$: $h(S) = 7$, y el costo real óptimo $c^*(S \to G) = 2 + 2 + 3 = 7$.
* En $A$: $h(A) = 5$, y el costo real óptimo $c^*(A \to G) = 2 + 3 = 5$.
* En $C$: $h(C) = 3$, y el costo real óptimo $c^*(C \to G) = 3$.
* En $G$: $h(G) = 0$.

Además, $h(n)$ es estrictamente decreciente a lo largo de esa rama ($7 \to 5 \to 3 \to 0$).
* **Comportamiento de Voraz:** Al evaluar exclusivamente $h(n)$, en $S$ prefiere $A$ ($h=5$) sobre $B$ ($h=7$). Luego en $A$, prefiere $C$ ($h=3$) sobre $D$ ($h=6$). Y en $C$ extrae directamente $G$ ($h=0$). La heurística "no miente", por lo que la codicia local de Voraz acierta directamente el camino óptimo.
* **Comportamiento de A\*:** En cada nodo de la rama óptima, $f(n) = g(n) + h(n) = 7$ (permanece constante e igual al costo óptimo $C^* = 7$). Como cualquier rama alternativa tiene un costo estimado estrictamente mayor ($B$ tiene $f = 2 + 7 = 9$; $D$ vía $A$ tiene $f = 7 + 6 = 13$), A* jamás se distrae explorando ramas laterales.

---

### Pregunta 2: ¿Garantiza Voraz devolver el camino de menor costo en general? Justificá con la propiedad de su prioridad (no con este ejemplo).

**Respuesta Técnica:**
**NO, Voraz no garantiza devolver el camino de menor costo.**

**Justificación formal:**  
La prioridad de Voraz es estrictamente $f(n) = h(n)$, lo que implica una evaluación puramente miope hacia el futuro, omitiendo de raíz el costo acumulado hacia atrás ($g(n)$).
* Formalmente: Sean dos estados sucesores $n_1$ y $n_2$. Si $h(n_1) < h(n_2)$, Voraz siempre expandirá primero $n_1$, independientemente de que $g(n_1) = 1.000.000$ y $g(n_2) = 1$.
* El algoritmo es susceptible a caer en "callejones sin salida prometedores" o en caminos locales donde cada paso parece estar cerca de la meta pero requiere atravesar aristas astronómicamente caras. Carece de la propiedad de **admisibilidad**, ya que no optimiza la función de costo total del camino $C$.

---

### Pregunta 3: ¿Qué ocurre si se usa $h = 0$ en A*? ¿Con qué algoritmo coincide entonces?

**Respuesta Técnica:**
Si fijamos la función heurística $h(n) = 0$ para todo estado $n \in S$:
$$f(n) = g(n) + h(n) = g(n) + 0 = g(n)$$

La función de evaluación de A* se reduce idénticamente a la función de prioridad de **Búsqueda de Costo Uniforme (UCS)** (equivalente al algoritmo de Dijkstra para grafos con aristas no negativas).
* Las curvas de nivel de la frontera pasan de ser elipses deformadas dirigidas hacia el objetivo a círculos concéntricos de isocosto $g$ que se expanden en todas direcciones desde el origen $S$.
* Si adicionalmente todas las aristas tuvieran costo uniforme constante ($c=1$), el algoritmo degeneraría exactamente en una Búsqueda en Amplitud (**BFS**).

---

### Pregunta 4: ¿Hubo reaperturas en UCS? ¿Y en A* y Voraz? ¿Por qué?

**Respuesta Técnica:**
* **En UCS:** **SÍ, hubo exactamente 1 reapertura en el estado $D$.**
  * *Detalle:* Al expandir el nodo $A$ ($g=2$), se generó $D$ a través de la arista $A \to D$ con costo $g = 2 + 5 = 7$, guardándose `mejor_g['D'] = 7`.
  * Posteriormente, al expandir $B$ ($g=2$), se descubrió la arista alternativa $B \to D$ ($c=2$), que produce un costo $g = 2 + 2 = 4$.
  * Como $4 < 7$, se detectó una mejora estricta del camino hacia un estado ya visitado/generado. Por lo tanto, se actualizó `mejor_g['D'] = 4` y se volvió a insertar $D$ en la frontera (reapertura).
* **En Voraz y A\*:** **NO hubo reaperturas (0 reaperturas).**
  * *Razón:* Tanto Voraz como A* fueron conducidos directamente hacia $C$ y $G$ por el gradiente de la heurística. En ambos casos, la meta $G$ fue alcanzada y extraída antes de que el nodo $B$ tuviese la oportunidad de ser expandido. Como $B$ nunca se expandió, la arista $B \to D$ jamás fue evaluada, imposibilitando cualquier descubrimiento de un nuevo camino a $D$.
  * Además, en el caso de A*, la heurística sobre el subgrafo explorado es **consistente (o monótona)**, cumpliendo la desigualdad triangular $h(n) \le c(n, n') + h(n')$. Teóricamente, A* con grafo y heurística consistente nunca reabre nodos ya cerrados.

---

### Pregunta 5: ¿"Expandir menos estados" significa "camino más barato"? Relacionalo con lo que muestran UCS y Voraz aquí.

**Respuesta Técnica:**
**NO, son conceptos completamente ortogonales:**
1. **Expandir estados** mide la **complejidad temporal y espacial (eficiencia computacional)** del proceso de búsqueda.
2. **El costo del camino** mide la **calidad de la solución encontrada (optimalidad)**.

**Relación con los resultados observados:**
* En este grafo particular, Voraz expandió únicamente **3 estados** ($S, A, C$) frente a los **5 estados** que expandió UCS ($S, A, B, C, D$), logrando ambos el mismo costo ($C=7$).
* Esta coincidencia es una particularidad de este diseño didáctico porque la heurística $h$ fue deliberadamente calibrada de forma exacta.
* Si alterásemos el grafo cambiando el costo de la arista $C \to G$ a $c=100$, Voraz seguiría expandiendo ciegamente $S \to A \to C \to G$ devolviendo un camino de costo pésimo ($104$) en solo 3 expansiones, mientras que UCS exploraría pacientemente más estados ($S, B, D, G$) para descubrir y devolver la ruta verdaderamente económica ($S \to B \to D \to G$ con costo $10$).
* **Conclusión:** Expandir menos estados simplemente significa que el algoritmo tomó decisiones más rápido; no da ninguna garantía sobre si esas decisiones fueron las correctas salvo que el algoritmo cuente con una base formal de optimalidad (como A* con heurística admisible).
